# [Improved Model] NFL Draft Prediction

This notebook builds on the baseline model with better feature engineering, target encoding, and a stronger model (LightGBM).

## Table of Contents
1. Setup
2. Load the Data
3. Data Analysis & EDA
4. Preprocessing
5. Feature Engineering
6. Model Training
   - 6.1 Why LightGBM?
   - 6.2 Cross-Validation Strategy
   - 6.3 Feature Importance
7. Hyperparameter Tuning (Optuna)
8. Stacking / Blending
9. School Conference Grouping
10. Create the Submission File

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# Install LightGBM if not already installed
# !pip install lightgbm

## 2. Load the Data

In [ ]:
train = pd.read_csv('input/train.csv', index_col='Id')
test  = pd.read_csv('input/test.csv',  index_col='Id')

print('Train shape:', train.shape)
print('Test shape: ', test.shape)
train.head()

## 3. Data Analysis & EDA

### 3.1 Target Distribution

First, let's check how balanced the target is.

In [ ]:
print('Draft rate:', train['Drafted'].mean().round(3))
train['Drafted'].value_counts().plot(kind='bar', title='Drafted Distribution')
plt.xticks([0, 1], ['Not Drafted (0)', 'Drafted (1)'], rotation=0)
plt.ylabel('Count')
plt.show()

### 3.2 Missing Values

Many physical test columns have missing values. Let's see how many.

In [ ]:
missing = train.isnull().mean().sort_values(ascending=False)
print(missing[missing > 0])

### 3.3 Draft Rate by Position Type

Some positions are drafted at much higher rates than others.

In [ ]:
draft_by_pos = train.groupby('Position_Type')['Drafted'].mean().sort_values(ascending=False)
draft_by_pos.plot(kind='bar', title='Draft Rate by Position Type')
plt.ylabel('Draft Rate')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 3.4 Physical Test vs Draft

Drafted players tend to run faster (lower 40-yard dash time).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
num_cols = ['Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
            'Broad_Jump', 'Agility_3cone', 'Shuttle']

for ax, col in zip(axes.flatten(), num_cols):
    train.boxplot(column=col, by='Drafted', ax=ax)
    ax.set_title(col)
    ax.set_xlabel('Drafted')

plt.suptitle('Physical Tests vs Drafted')
plt.tight_layout()
plt.show()

### 3.5 School Draft Rate

The School column has 236 unique values, but it carries useful signal — players from top programs get drafted more often. Instead of dropping it (as the baseline does), we will use **target encoding** to capture this signal.

In [ ]:
school_rate = (
    train.groupby('School')['Drafted']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'Draft_Rate', 'count': 'N'})
    .query('N >= 10')
    .sort_values('Draft_Rate', ascending=False)
)
print('Top 10 schools by draft rate:')
print(school_rate.head(10))
print('\nBottom 10 schools by draft rate:')
print(school_rate.tail(10))

## 4. Preprocessing

### 4.1 Separate Target

In [ ]:
target = train['Drafted'].copy()
train  = train.drop(columns=['Drafted'])

print('Target shape:', target.shape)
print('Train shape: ', train.shape)

### 4.1b Position-Specific Imputation

Instead of filling missing combine values with a **global median**, we fill them with the **median for that specific Position**.

A DT and a CB have very different physical profiles — using a single global median for Sprint_40yd would give a CB the median of an OT, which is wrong.

We compute position medians from the **training set only** to avoid leakage into the test set.

In [ ]:
combine_cols = ['Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
                'Broad_Jump', 'Agility_3cone', 'Shuttle']

# Compute position-level medians from train only (no leakage)
pos_medians    = train.groupby('Position')[combine_cols].median()
global_medians = train[combine_cols].median()

def impute_by_position(df, pos_medians, global_medians):
    df = df.copy()
    for col in combine_cols:
        # Map each row's Position to its position-level median
        pos_fill = df['Position'].map(pos_medians[col])
        # Fall back to global median for unseen positions
        df[col] = df[col].fillna(pos_fill).fillna(global_medians[col])
    return df

train = impute_by_position(train, pos_medians, global_medians)
test  = impute_by_position(test,  pos_medians, global_medians)

print('Position-specific imputation done.')
print('Remaining nulls in combine cols:')
print(train[combine_cols].isnull().sum())

### 4.1c Position-Relative Features

How good is a player **relative to others at the same position**?

A 4.5s 40-yard dash is elite for an OT but slow for a CB. By computing a **z-score** per position, we tell the model how exceptional each player is within their position group — which is exactly how scouts think.

Formula: `z = (value - position_mean) / position_std`

We compute position stats from the **training set only** and apply them to both train and test.

In [ ]:
# Compute position mean and std from train only (no leakage)
pos_stats = train.groupby('Position')[combine_cols].agg(['mean', 'std'])

def add_position_zscores(df, pos_stats):
    df = df.copy()
    for col in combine_cols:
        pos_mean = df['Position'].map(pos_stats[col]['mean'])
        pos_std  = df['Position'].map(pos_stats[col]['std'])
        df[f'{col}_zscore'] = (df[col] - pos_mean) / (pos_std + 1e-6)
    return df

train = add_position_zscores(train, pos_stats)
test  = add_position_zscores(test,  pos_stats)

zscore_cols = [f'{c}_zscore' for c in combine_cols]
print('Position z-score features added:')
print(train[zscore_cols].describe().round(2))

### 4.1d Missing Value Count Feature

Players with many missing combine measurements are often specialists or players who skipped the combine. This missingness pattern is itself predictive.

In [ ]:
train['Missing_Count'] = train[combine_cols].isnull().sum(axis=1)
test['Missing_Count']  = test[combine_cols].isnull().sum(axis=1)

print('Missing count distribution in train:')
print(train['Missing_Count'].value_counts().sort_index())
print('\nDraft rate by missing count:')
print(train.assign(Drafted=target).groupby('Missing_Count')['Drafted'].mean().round(3))

### 4.2 Target Encoding for School

Instead of dropping School, we replace each school name with its historical draft rate.

**Why out-of-fold (OOF)?**  
If we compute the draft rate using the same rows we train on, the model will memorise the target — this is called **data leakage**. OOF encoding computes the rate on held-out folds only, so the encoding is honest.

In [ ]:
school_te = np.zeros(len(train))
skf_te    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for tr_idx, val_idx in skf_te.split(train, target):
    # Compute draft rate per school using only the training fold
    school_mean = (
        train.iloc[tr_idx]
        .assign(Drafted=target.iloc[tr_idx])
        .groupby('School')['Drafted']
        .mean()
    )
    # Fill validation fold with the computed rate (unknown schools get global mean)
    school_te[val_idx] = (
        train.iloc[val_idx]['School']
        .map(school_mean)
        .fillna(target.mean())
    )

train['School_TE'] = school_te

# For test: use the full training set mean
school_mean_full = (
    train.assign(Drafted=target)
    .groupby('School')['Drafted']
    .mean()
)
test['School_TE'] = test['School'].map(school_mean_full).fillna(target.mean())

# Drop the raw School column
train = train.drop(columns=['School'])
test  = test.drop(columns=['School'])

print('School_TE sample:')
print(train['School_TE'].describe())

### 4.3 Target Encoding for Position

Label encoding treats positions as arbitrary integers. Target encoding replaces each position with its historical draft rate, so the model knows QBs get drafted at a different rate than Ks. Same OOF approach as School to avoid leakage.

In [ ]:
position_te = np.zeros(len(train))
skf_pos     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for tr_idx, val_idx in skf_pos.split(train, target):
    pos_mean = (
        train.iloc[tr_idx]
        .assign(Drafted=target.iloc[tr_idx])
        .groupby('Position')['Drafted']
        .mean()
    )
    position_te[val_idx] = (
        train.iloc[val_idx]['Position']
        .map(pos_mean)
        .fillna(target.mean())
    )

train['Position_TE'] = position_te

pos_mean_full = (
    train.assign(Drafted=target)
    .groupby('Position')['Drafted']
    .mean()
)
test['Position_TE'] = test['Position'].map(pos_mean_full).fillna(target.mean())

print('Draft rate by position:')
print(pos_mean_full.sort_values(ascending=False).round(3))

### 4.4 Label Encoding for Categorical Columns

We encode `Player_Type`, `Position_Type`, and `Position` as integers so the model can use them.

In [ ]:
cat_cols = ['Player_Type', 'Position_Type', 'Position']
le = LabelEncoder()

for col in cat_cols:
    combined = pd.concat([train[col], test[col]], axis=0).astype(str)
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))

print('Categorical columns encoded.')
train[cat_cols].head()

## 5. Feature Engineering

We create new features that capture athletic ability in ways the raw numbers do not.

| Feature | Formula | Why it helps |
|---|---|---|
| BMI | Weight / Height² | Body composition |
| Speed Score | Weight × 200 / Sprint_40yd⁴ | NFL standard: big fast players score higher |
| Burst Score | Vertical + Broad Jump | Explosive power |
| Agility Score | 3cone + Shuttle | Change-of-direction ability |
| Weight-Height Ratio | Weight / Height | Size relative to height |
| Year Recency | Year − 2009 | Scouting trends change over time |
| Age Missing Flag | 1 if Age is NaN | Missing age may itself be informative |

In [ ]:
def add_features(df):
    df = df.copy()

    # BMI
    df['BMI'] = df['Weight'] / (df['Height'] ** 2)

    # Speed Score — standard NFL combine metric
    df['Speed_Score'] = (df['Weight'] * 200) / (df['Sprint_40yd'] ** 4)

    # Burst Score — explosive power
    df['Burst_Score'] = df['Vertical_Jump'] + df['Broad_Jump']

    # Agility Score — lower is better, so higher sum = worse agility
    df['Agility_Score'] = df['Agility_3cone'] + df['Shuttle']

    # Athletic Composite — combined jump and strength score
    df['Athletic_Composite'] = (
        df['Vertical_Jump'].fillna(df['Vertical_Jump'].median()) +
        df['Broad_Jump'].fillna(df['Broad_Jump'].median()) / 10 +
        df['Bench_Press_Reps'].fillna(df['Bench_Press_Reps'].median())
    )

    # Weight-to-height ratio
    df['Weight_Height_Ratio'] = df['Weight'] / df['Height']

    # Year recency
    df['Year_Recency'] = df['Year'] - 2009

    # Age missing flag
    df['Age_Missing'] = df['Age'].isna().astype(int)

    return df

train = add_features(train)
test  = add_features(test)

print('New features added. Train shape:', train.shape)
print('New columns:', ['BMI','Speed_Score','Burst_Score','Agility_Score',
                       'Athletic_Composite','Weight_Height_Ratio',
                       'Year_Recency','Age_Missing'])

### 5.1 Check New Features vs Draft

Let's confirm the new features separate drafted from non-drafted players.

In [ ]:
new_features = ['BMI', 'Speed_Score', 'Burst_Score', 'Agility_Score']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, new_features):
    train.assign(Drafted=target).boxplot(column=col, by='Drafted', ax=ax)
    ax.set_title(col)
    ax.set_xlabel('Drafted')

plt.suptitle('New Features vs Drafted')
plt.tight_layout()
plt.show()

## 6. Model Training

### 6.1 Why LightGBM?

The baseline uses a Random Forest. We switch to **LightGBM** because:
- It handles missing values natively (no need to impute)
- It is faster and usually achieves higher AUC on tabular data
- It supports early stopping to avoid overfitting

### 6.2 Cross-Validation Strategy

We use **5-fold Stratified K-Fold** — the same as the baseline — to get a reliable estimate of AUC.

In [ ]:
feature_cols = list(train.columns)
print(f'Total features: {len(feature_cols)}')
print(feature_cols)

In [ ]:
lgb_params = {
    'objective':         'binary',
    'metric':            'auc',
    'learning_rate':     0.05,
    'num_leaves':        63,
    'max_depth':         -1,
    'min_child_samples': 20,
    'feature_fraction':  0.8,
    'bagging_fraction':  0.8,
    'bagging_freq':      5,
    'reg_alpha':         0.1,
    'reg_lambda':        0.1,
    'verbose':           -1,
    'random_state':      42,
}

oof_preds  = np.zeros(len(train))
test_preds = np.zeros(len(test))
n_splits   = 5
skf        = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold, (tr_idx, val_idx) in enumerate(skf.split(train, target)):
    X_tr,  y_tr  = train.iloc[tr_idx][feature_cols],  target.iloc[tr_idx]
    X_val, y_val = train.iloc[val_idx][feature_cols],  target.iloc[val_idx]

    dtrain = lgb.Dataset(X_tr,  label=y_tr)
    dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)

    model = lgb.train(
        lgb_params,
        dtrain,
        num_boost_round=1000,
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(200)
        ],
    )

    oof_preds[val_idx]  = model.predict(X_val[feature_cols])
    test_preds         += model.predict(test[feature_cols]) / n_splits

    fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
    print(f'Fold {fold+1}  AUC: {fold_auc:.4f}  |  Best iteration: {model.best_iteration}')

overall_auc = roc_auc_score(target, oof_preds)
print(f'\nOverall OOF AUC: {overall_auc:.4f}')

### 6.3 Feature Importance

Which features does the model rely on most?

In [ ]:
importance = pd.Series(
    model.feature_importance(importance_type='gain'),
    index=feature_cols
).sort_values(ascending=False)

plt.figure(figsize=(10, 8))
importance.head(20).plot(kind='barh')
plt.title('Top 20 Feature Importances (Gain)')
plt.xlabel('Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Hyperparameter Tuning (Optuna)

The LightGBM params in Section 6 are reasonable defaults. Optuna searches for better ones automatically by running many CV trials and keeping the best.

Install if needed: `!pip install optuna`

In [ ]:
# !pip install optuna
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'objective':         'binary',
        'metric':            'auc',
        'verbose':           -1,
        'random_state':      42,
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 31, 127),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'feature_fraction':  trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction':  trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq':      5,
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-3, 1.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-3, 1.0, log=True),
    }
    skf_opt = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof = np.zeros(len(train))
    for tr_idx, val_idx in skf_opt.split(train, target):
        X_tr, y_tr   = train.iloc[tr_idx][feature_cols], target.iloc[tr_idx]
        X_val, y_val = train.iloc[val_idx][feature_cols], target.iloc[val_idx]
        dtrain = lgb.Dataset(X_tr, label=y_tr)
        dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)
        m = lgb.train(params, dtrain, num_boost_round=1000, valid_sets=[dval],
                      callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
        oof[val_idx] = m.predict(X_val[feature_cols])
    return roc_auc_score(target, oof)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print('Best AUC:', study.best_value)
print('Best params:', study.best_params)

## 8. Stacking / Blending

Train a second model (XGBoost) with the same CV setup and average the predictions with LightGBM. Even a simple 50/50 blend often improves AUC by 0.002–0.005 because the two models make different errors.

Install if needed: `!pip install xgboost`

In [ ]:
# !pip install xgboost
import xgboost as xgb

xgb_params = {
    'objective':        'binary:logistic',
    'eval_metric':      'auc',
    'learning_rate':    0.05,
    'max_depth':        6,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'reg_alpha':        0.1,
    'reg_lambda':       0.1,
    'seed':             42,
    'verbosity':        0,
}

oof_xgb  = np.zeros(len(train))
test_xgb = np.zeros(len(test))
skf_xgb  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (tr_idx, val_idx) in enumerate(skf_xgb.split(train, target)):
    X_tr,  y_tr  = train.iloc[tr_idx][feature_cols],  target.iloc[tr_idx]
    X_val, y_val = train.iloc[val_idx][feature_cols], target.iloc[val_idx]
    dtrain = xgb.DMatrix(X_tr,  label=y_tr)
    dval   = xgb.DMatrix(X_val, label=y_val)
    m = xgb.train(xgb_params, dtrain, num_boost_round=1000,
                  evals=[(dval, 'val')], early_stopping_rounds=50, verbose_eval=False)
    oof_xgb[val_idx] = m.predict(dval)
    test_xgb        += m.predict(xgb.DMatrix(test[feature_cols])) / 5
    print(f'Fold {fold+1} XGB AUC: {roc_auc_score(y_val, oof_xgb[val_idx]):.4f}')

print(f'\nXGB OOF AUC: {roc_auc_score(target, oof_xgb):.4f}')

# Blend 50/50
blend_oof   = 0.5 * oof_preds  + 0.5 * oof_xgb
blend_test  = 0.5 * test_preds + 0.5 * test_xgb
print(f'Blended OOF AUC: {roc_auc_score(target, blend_oof):.4f}')

## 9. School Conference Grouping

Power 5 conference schools (SEC, Big Ten, ACC, Big 12, Pac-12) produce far more draftees than Group of 5 or FCS schools. Adding a conference tier feature gives the model this domain knowledge directly.

In [ ]:
# Map schools to conference tiers
sec    = ['Alabama','LSU','Georgia','Florida','Auburn','Tennessee','Mississippi',
          'Mississippi St.','Arkansas','South Carolina','Kentucky','Vanderbilt',
          'Missouri','Texas A&M']
big10  = ['Ohio St.','Michigan','Penn St.','Wisconsin','Iowa','Michigan St.',
          'Nebraska','Minnesota','Northwestern','Indiana','Purdue','Illinois',
          'Maryland','Rutgers']
acc    = ['Clemson','Florida St.','Miami (FL)','Virginia Tech','North Carolina',
          'North Carolina St.','Boston Col.','Virginia','Georgia Tech','Pittsburgh',
          'Syracuse','Wake Forest','Louisville','Duke']
big12  = ['Oklahoma','Texas','Baylor','TCU','Oklahoma St.','Kansas St.',
          'Iowa St.','West Virginia','Kansas','Texas Tech']
pac12  = ['USC','UCLA','Oregon','Washington','Stanford','California',
          'Arizona St.','Arizona','Utah','Colorado','Oregon St.','Washington St.']

def get_conference_tier(school):
    if school in sec:   return 3  # SEC — highest draft rate
    if school in big10: return 3  # Big Ten
    if school in acc:   return 2  # ACC
    if school in big12: return 2  # Big 12
    if school in pac12: return 2  # Pac-12
    return 1                      # Group of 5 / FCS / other

# Apply before School column is dropped (run this cell before 4.2)
# If School is already dropped, reload and re-run from 4.1
if 'School' in train.columns:
    train['Conference_Tier'] = train['School'].map(get_conference_tier)
    test['Conference_Tier']  = test['School'].map(get_conference_tier)
else:
    print('School column already dropped. Re-run from Section 4.1 with this cell added before 4.2.')

print('Conference tier distribution:')
print(train['Conference_Tier'].value_counts())
print('\nDraft rate by conference tier:')
print(train.assign(Drafted=target).groupby('Conference_Tier')['Drafted'].mean().round(3))

## 10. Create the Submission File

In [ ]:
submission = pd.read_csv('input/sample_submission.csv')
submission['Drafted'] = test_preds
submission.to_csv('submission.csv', index=False)

print('submission.csv saved!')
print(submission.head())
print(f'\nPrediction range: {test_preds.min():.4f} — {test_preds.max():.4f}')

## Next Steps

Here are directions to improve further:

- **Interaction features** — e.g. `Speed_Score × Position_Type` to capture that speed matters differently by position
- **Hyperparameter tuning** — use Optuna or GridSearchCV to find better LightGBM parameters
- **Stacking** — combine LightGBM predictions with Random Forest or XGBoost predictions
- **School grouping** — group schools by conference (SEC, Big Ten, etc.) as an additional feature